# 🗂️ Notebook 2: Gmail — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

```
users(id, email, ...)
mailboxes(user_id → has many messages)

messages
   id (message-id, guid)
   user_id         -- partition key
   thread_id       -- messages grouped into conversations
   from_addr, to_addrs[], cc, bcc
   subject
   body_storage_url   -- points to object store
   size_bytes
   received_at
   labels            -- "inbox", "sent", "starred", custom
   is_read, is_spam

threads
   id, user_id, subject, last_msg_at, msg_count

attachments
   id, message_id, filename, content_type, storage_url
```

### Threading
Gmail groups messages into a **thread** (conversation) using:
- Matching `References` / `In-Reply-To` headers.
- Fallback on normalized subject (`Re: Foo` → same thread as `Foo`).

## Core APIs

```http
GET  /messages?label=inbox&limit=50&page_token=...
GET  /messages/{id}                → full message
POST /messages                     → send (goes to outbound SMTP)
POST /messages/{id}/labels         → { add: [...], remove: [...] }
GET  /search?q=from:alice attachment:yes has:link after:2025/01/01
```


In [ ]:
from pydantic import BaseModel, EmailStr
from datetime import datetime
class SendEmail(BaseModel):
    to: list[str]
    subject: str
    body: str
    cc: list[str] = []
print(SendEmail(to=["alice@example.com"], subject="hi", body="hello").model_dump_json(indent=2))
